# Notebook 01 — Data Collection

## Objectives
- Fetch the used-car dataset from Kaggle (our data **endpoint**) using the Kaggle API.
- Combine the per-manufacturer files into a single dataset, handling known data quirks.
- Save the combined dataset and take a first look (CRISP-DM: Data Understanding).

## Inputs
- Kaggle authentication token, read automatically from `~/.kaggle/access_token`
  (kept in the home directory, **never committed** to the repo).
- Kaggle dataset: `adityadesai13/used-car-dataset-ford-and-mercedes`
  (100,000 UK Used Car Data set, licence **CC0: Public Domain**).

## Outputs
- Raw CSVs saved to `inputs/datasets/raw/`.
- A single combined dataset saved to `outputs/datasets/collection/used_cars.csv`.

> Satisfies Pass **7.1** (data collection from an endpoint via Jupyter Notebook)
> and contributes to Pass **1.1** (Business/Data Understanding under CRISP-DM).

---
## 1. Imports and working directory

In [ ]:
import os
import pandas as pd

# Make sure we run from the project root, whether the notebook is opened
# from the root or from inside the jupyter_notebooks/ folder.
if os.path.basename(os.getcwd()) == "jupyter_notebooks":
    os.chdir(os.path.dirname(os.getcwd()))
print("Working directory:", os.getcwd())

---
## 2. Fetch the dataset from Kaggle (the endpoint)

The Kaggle client reads the API token automatically from `~/.kaggle/access_token`.
We download the dataset and unzip it straight into `inputs/datasets/raw/`.

In [ ]:
DATASET = "adityadesai13/used-car-dataset-ford-and-mercedes"
RAW_DIR = "inputs/datasets/raw"
os.makedirs(RAW_DIR, exist_ok=True)

!kaggle datasets download -d {DATASET} -p {RAW_DIR} --unzip

In [ ]:
# Confirm what was downloaded
sorted(f for f in os.listdir(RAW_DIR) if f.endswith(".csv"))

---
## 3. Combine the manufacturer files

The dataset ships one CSV per manufacturer. To build a multi-brand pricing tool we
combine them into one dataframe with a `manufacturer` column. Before combining we:

- **Skip** the model-subset files (`cclass.csv`, `focus.csv`) — their cars already
  appear inside `merc.csv` and `ford.csv`, so including them would duplicate rows.
- **Skip** the `unclean ...` files — they are the raw, pre-cleaned versions.
- **Standardise columns** — the Hyundai file uses `tax(£)`; we rename it to `tax`
  so every file lines up.

In [ ]:
# Files that must NOT be combined (duplicates / pre-cleaned versions)
SKIP = {"cclass.csv", "focus.csv", "unclean cclass.csv", "unclean focus.csv"}

# Tidy display names for each manufacturer file
MANUFACTURER_NAMES = {
    "audi": "Audi", "bmw": "BMW", "ford": "Ford", "hyundi": "Hyundai",
    "merc": "Mercedes", "skoda": "Skoda", "toyota": "Toyota",
    "vauxhall": "Vauxhall", "vw": "Volkswagen",
}

frames = []
for f in sorted(os.listdir(RAW_DIR)):
    if f.endswith(".csv") and f not in SKIP:
        tmp = pd.read_csv(os.path.join(RAW_DIR, f))
        tmp.columns = [col.strip() for col in tmp.columns]   # drop stray whitespace
        tmp = tmp.rename(columns={"tax(£)": "tax"})          # align the Hyundai quirk
        key = f.replace(".csv", "")
        tmp["manufacturer"] = MANUFACTURER_NAMES.get(key, key.title())
        frames.append(tmp)

df = pd.concat(frames, ignore_index=True)
print(f"Combined {len(frames)} manufacturer files into one dataset.")
df.shape

---
## 4. First look (Data Understanding)

A quick inspection: column types, a sample of rows, summary statistics, and a
missing-value check. We will write our conclusions from these outputs into the
README and the Project Summary dashboard page.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Missing values per column
df.isna().sum()

In [ ]:
# How many rows per manufacturer?
df["manufacturer"].value_counts()

---
## 5. Save the combined dataset

In [ ]:
OUT_DIR = "outputs/datasets/collection"
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, "used_cars.csv")
df.to_csv(out_path, index=False)
print("Saved:", out_path, "| rows:", len(df), "| columns:", df.shape[1])

---
## 6. Conclusions and next steps
- _TODO (together, once we see the output): summarise rows, columns, what each
  variable means, dtype issues, and any missing values._
- Next: **Notebook 02 — Data Cleaning** (CRISP-DM: Data Preparation).